In [4]:
import torch
import numpy as np
from torch.utils.data import Dataset # 引入数据集
from torch.utils.data import DataLoader # 引入数据加载集
# https://blog.csdn.net/bit452/article/details/109686474
# https://blog.csdn.net/weixin_44841652/article/details/105129235#comments_13684803

### DataSet 是抽象类，不能实例化对象，主要是用于构造我们的数据集
### DataLoader 需要获取DataSet提供的索引[i]和len;用来帮助我们加载数据，比如说做shuffle(提高数据集的随机性)，batch_size,能拿出Mini-Batch进行训练。它帮我们自动完成这些工作。DataLoader可实例化对象。
### __getitem__目的是为支持下标(索引)操作,__len__是获取长度。都是magic function

1. 需要mini_batch 就需要import DataSet和DataLoader

2. 继承DataSet的类需要重写init，getitem,len魔法函数。分别是为了加载数据集，获取数据索引，获取数据总量。

3. DataLoader对数据集先打乱(shuffle)，然后划分成mini_batch。

4. len函数的返回值 除以 batch_size 的结果就是每一轮epoch中需要迭代的次数。

5. inputs, labels = data中的inputs的shape是[32,8],labels 的shape是[32,1]。也就是说mini_batch在这个地方体现的

6. diabetes.csv数据集老师给了下载地址，该数据集需和源代码放在同一个文件夹内。


In [5]:
# prepare dataset
'''
Dataset是一个抽象函数，不能直接实例化，所以我们要创建一个自己类，继承Dataset
继承Dataset后我们必须实现三个函数：
__init__()是初始化函数，之后我们可以提供数据集路径进行数据的加载
__getitem__()帮助我们通过索引找到某个样本
__len__()帮助我们返回数据集大小

'''
class DiabetesDataset(Dataset):
    def __init__(self, filpath):
        xy = np.loadtxt(filpath, delimiter=',', dtype=np.float32)
        self.len = xy.shape[0] # shape(行，列)
        ''' 
            shape本身是一个二元组(x,y)，对应数据集的行数和列数，这里[0]表示取行数
            shpae[0] ==> 行数
            shape[1] ==> 列数
        '''
        self.x_data = torch.from_numpy(xy[: , : -1])
        self.y_data = torch.from_numpy(xy[: , [-1]])

    def __getitem__(self, index):
        return self.x_data[index], self.y_data[index]
    
    def __len__(self):
        return self.len
    
dataset = DiabetesDataset("diabetes.csv")
train_loader = DataLoader(dataset = dataset, batch_size = 32, shuffle = True, num_workers=0) # num_workers 多线程
# 分组
#   dataset选择要进行的数据集
#   batch_size选择mini-batch每一次批处理的大小
#   shuffle选择是否进行随机排序
#   num_workers是否使用多线程
''' 

'''

# design model using class
class Model(torch.nn.Module):
    def __init__(self):
        super(Model, self).__init__()
        self.linear1 = torch.nn.Linear(8, 6)
        self.linear2 = torch.nn.Linear(6, 4)
        self.linear3 = torch.nn.Linear(4, 1)
        self.sigmoid = torch.nn.Sigmoid()

    def forward(self, x): # 构造计算图
        x = self.sigmoid(self.linear1(x))
        x = self.sigmoid(self.linear2(x))
        x = self.sigmoid(self.linear3(x))
        return x
    
model = Model()

# construct loss and optimizer
criterion = torch.nn.BCELoss(reduction = 'mean')
optimizer = torch.optim.SGD(model.parameters(), lr = 0.01)
#model.parameters()会扫描module中的所有成员，如果成员中有相应权重，那么都会将结果加到要训练的参数集合上

# training cycle
if __name__ == '__main__': # #if这条语句在windows系统下一定要加，否则会报错
    for epoch in range(100):
        for i, data in enumerate(train_loader, 0):
            # Prepare data
            inputs, labels = data # inputs ==> x , labels ==> y
            # Forward
            y_pred = model(inputs)
            loss = criterion(y_pred, labels)
            print(epoch, i, loss.item())

            # Backward
            optimizer.zero_grad()
            loss.backward()

            # Update
            optimizer.step()



0 0 0.680785596370697
0 1 0.6811909079551697
0 2 0.6827882528305054
0 3 0.6807136535644531
0 4 0.6666785478591919
0 5 0.68581223487854
0 6 0.687980055809021
0 7 0.6812190413475037
0 8 0.6888111233711243
0 9 0.6855400204658508
0 10 0.6831780076026917
0 11 0.6800329089164734
0 12 0.6797832250595093
0 13 0.664654552936554
0 14 0.6911635994911194
0 15 0.6754201650619507
0 16 0.684177815914154
0 17 0.6840004324913025
0 18 0.6780589818954468
0 19 0.6907992959022522
0 20 0.6710017323493958
0 21 0.6669301390647888
0 22 0.6768527626991272
0 23 0.6815900206565857
1 0 0.6945605874061584
1 1 0.6794161200523376
1 2 0.6574496626853943
1 3 0.6785285472869873
1 4 0.6708882451057434
1 5 0.6907484531402588
1 6 0.6822278499603271
1 7 0.6660834550857544
1 8 0.6826545000076294
1 9 0.686527669429779
1 10 0.6861358880996704
1 11 0.6653818488121033
1 12 0.6465846300125122
1 13 0.663203775882721
1 14 0.6765789985656738
1 15 0.6675833463668823
1 16 0.6956702470779419
1 17 0.6762167811393738
1 18 0.6812601089477